### Discussion: train_ce20_arc80 + infer_ce50_arc50

In [ ]:
%%time
# =========================================================
# ArcFace train-grid × infer-grid × mixing-mode sweep
# ---------------------------------------------------------
# This script is designed as an experimental sweep driver.
# It trains ArcFace/CE models under one or more training-loss ratios,
# then evaluates one or more inference-time CE/ArcFace logit fusion ratios.
#
# Compare:
#   1) raw mixing
#   2) fixed-scale normalized mixing
#   3) EMA normalized mixing
#
# Outputs:
#   ./ensemble_artifacts/traingrid_norm/{train_name}__{mix_tag}/best_fold{fold}.pth
#   ./ensemble_artifacts/traingrid_norm/{train_name}__{mix_tag}/{infer_name}_oof_probs.npy
#   ./ensemble_artifacts/traingrid_norm/{train_name}__{mix_tag}/{infer_name}_test_probs.npy
#   ./ensemble_artifacts/traingrid_norm/{train_name}__{mix_tag}/{infer_name}_submission.csv
#   ./ensemble_artifacts/traingrid_norm_summary.csv
# =========================================================

# !pip -q install timm albumentations opencv-python-headless

import os
import gc
import cv2
import math
import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

# =========================================================
# TRAIN GRID / INFER GRID / MIXING MODES
# =========================================================
# Each training setting specifies the relative weight of standard CE loss
# and ArcFace loss during optimization. Commented entries are kept as
# reusable alternatives for future experiments.
TRAIN_SETTINGS = [
    #{"name": "train_ce_only",    "train_ce": 1.0, "train_arc": 0.0},
    {"name": "train_ce20_arc80", "train_ce": 0.2, "train_arc": 0.8},
    #{"name": "train_ce40_arc60", "train_ce": 0.4, "train_arc": 0.6},
    #{"name": "train_ce80_arc20", "train_ce": 0.8, "train_arc": 0.2},
    #{"name": "train_ce60_arc40", "train_ce": 0.6, "train_arc": 0.4},
    #{"name": "train_ce50_arc50", "train_ce": 0.5, "train_arc": 0.5},
    #{"name": "train_arc_only",   "train_ce": 0.0, "train_arc": 1.0},
]

# Each inference setting specifies how CE logits and ArcFace logits are
# fused after training. Since inference settings are evaluated after
# checkpoint training, multiple fusion ratios can be tested without retraining.
INFER_SETTINGS = [
    #{"name": "infer_ce_only",    "infer_ce": 1.0, "infer_arc": 0.0},
    #{"name": "infer_ce90_arc10", "infer_ce": 0.9, "infer_arc": 0.1},
    #{"name": "infer_ce80_arc20", "infer_ce": 0.8, "infer_arc": 0.2},
    #{"name": "infer_ce60_arc40", "infer_ce": 0.6, "infer_arc": 0.4},
    {"name": "infer_ce50_arc50", "infer_ce": 0.5, "infer_arc": 0.5},
    #{"name": "infer_ce40_arc60", "infer_ce": 0.4, "infer_arc": 0.6},
    #{"name": "infer_ce20_arc80", "infer_ce": 0.2, "infer_arc": 0.8},
    #{"name": "infer_arc_only",   "infer_ce": 0.0, "infer_arc": 1.0},
]

# Loss mixing modes. The raw mode keeps the original loss magnitudes,
# while the normalized modes are retained for comparing scale-corrected objectives.
MIXING_MODES = [
    {
        "mix_tag": "raw",
        "mix_mode": "raw",
        "arc_loss_scale": 1.0,
        "ema_momentum": 0.95,
    },
    #{
    #    "mix_tag": "norm_fixed10",
    #    "mix_mode": "norm_fixed",
    #    "arc_loss_scale": 10.0,
    #    "ema_momentum": 0.95,
    #},
    #{
    #    "mix_tag": "norm_ema",
    #    "mix_mode": "norm_ema",
    #    "arc_loss_scale": 1.0,
    #    "ema_momentum": 0.95,
    #},
]

# Optional smaller grid for quick smoke tests. It is disabled by default
# and does not affect the main experimental setting unless FAST_DEBUG=True.
FAST_DEBUG = False
if FAST_DEBUG:
    TRAIN_SETTINGS = [
        {"name": "train_ce_only",    "train_ce": 1.0, "train_arc": 0.0},
        {"name": "train_ce60_arc40", "train_ce": 0.6, "train_arc": 0.4},
    ]
    INFER_SETTINGS = [
        {"name": "infer_ce_only",    "infer_ce": 1.0, "infer_arc": 0.0},
        {"name": "infer_ce50_arc50", "infer_ce": 0.5, "infer_arc": 0.5},
        {"name": "infer_arc_only",   "infer_ce": 0.0, "infer_arc": 1.0},
    ]
    MIXING_MODES = [
        {"mix_tag": "raw",         "mix_mode": "raw",        "arc_loss_scale": 1.0,  "ema_momentum": 0.95},
        {"mix_tag": "norm_fixed10","mix_mode": "norm_fixed", "arc_loss_scale": 10.0, "ema_momentum": 0.95},
    ]

# =========================================================
# Config
# =========================================================
# Central configuration object. It stores model, data, optimization,
# augmentation, and output settings, and is serialized into metadata files.
@dataclass
class CFG:
    seed: int = 42
    num_classes: int = 10

    data_root: str = "./"
    train_csv: str = "./Data/training.csv"
    test_csv: str = "./Data/test.csv"

    save_root: str = "./ensemble_artifacts"
    sweep_name: str = "traingrid_norm"

    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"

    image_size: int = 384
    embedding_dim: int = 512

    n_splits: int = 5
    epochs: int = 20

    train_bs: int = 16
    valid_bs: int = 32
    num_workers: int = 0

    lr_backbone: float = 1e-4
    lr_head: float = 2e-4
    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    arc_s: float = 30.0
    arc_m: float = 0.30

    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0
    tta_count: int = 4

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()

SAVE_DIR = os.path.join(cfg.save_root, cfg.sweep_name)
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================================================
# Utils
# =========================================================
def set_seed(seed=42):
    # Seed Python, NumPy, and PyTorch RNGs to make fold splits and training
    # as reproducible as possible under the same environment.
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

def make_abs_path(p):
    # Convert relative paths in the CSV files into paths under cfg.data_root.
    # Absolute paths are left unchanged for portability across local/Kaggle setups.
    if os.path.isabs(p):
        return p
    return os.path.join(cfg.data_root, p)

def exp_name(train_name, mix_tag):
    # Compose a stable experiment name from the training ratio and mixing mode.
    return f"{train_name}__{mix_tag}"

def exp_dir(train_name, mix_tag):
    # Return the output directory for one train/mix configuration and create it if needed.
    d = os.path.join(SAVE_DIR, exp_name(train_name, mix_tag))
    os.makedirs(d, exist_ok=True)
    return d

def best_model_path(train_name, mix_tag, fold):
    # Path to the best checkpoint for a single fold.
    return os.path.join(exp_dir(train_name, mix_tag), f"best_fold{fold}.pth")

def infer_oof_probs_path(train_name, mix_tag, infer_name):
    # Path to OOF probabilities for a specific inference fusion setting.
    return os.path.join(exp_dir(train_name, mix_tag), f"{infer_name}_oof_probs.npy")

def infer_test_probs_path(train_name, mix_tag, infer_name):
    # Path to fold-averaged test probabilities for a specific inference fusion setting.
    return os.path.join(exp_dir(train_name, mix_tag), f"{infer_name}_test_probs.npy")

def infer_submission_path(train_name, mix_tag, infer_name):
    # Path to the submission CSV generated from the corresponding test probabilities.
    return os.path.join(exp_dir(train_name, mix_tag), f"{infer_name}_submission.csv")

def infer_fold_scores_path(train_name, mix_tag, infer_name):
    # Path to per-fold validation metrics for a specific inference setting.
    return os.path.join(exp_dir(train_name, mix_tag), f"fold_scores_{infer_name}.csv")

def train_metadata_path(train_name, mix_tag):
    # Path to training metadata, including fold logs and best validation scores.
    return os.path.join(exp_dir(train_name, mix_tag), "train_metadata.json")

class EarlyStopping:
    # Minimal early-stopping helper. It tracks the best score and stops when
    # validation accuracy has not improved for cfg.patience epochs.
    def __init__(self, patience=4, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        improved = score > self.best_score + self.min_delta if self.mode == "max" else score < self.best_score - self.min_delta
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

# =========================================================
# Read data
# =========================================================
train_df = pd.read_csv(cfg.train_csv)
test_df = pd.read_csv(cfg.test_csv)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("train.csv must contain 'y' or 'TARGET'.")

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

# =========================================================
# Transforms
# =========================================================
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def get_train_transforms(img_size=384):
    # Training augmentations simulate moderate post-processing variations such as
    # crop/resize, blur, compression, grayscale conversion, and small rotations.
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.75, 1.0), ratio=(0.90, 1.10), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.15),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.ToGray(p=0.07),
        A.OneOf([
            A.Downscale(scale_range=(0.5, 0.85), p=1.0),
            A.Resize(int(img_size * 0.85), int(img_size * 0.85), p=1.0),
        ], p=0.12),
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size=384):
    # Deterministic validation preprocessing. It avoids random augmentation so
    # epoch-level validation metrics are comparable across epochs.
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size=384, tta_id=0):
    # Test-time augmentation variants. Predictions from all variants are averaged
    # in predict_with_tta() to reduce sensitivity to flips and small crops.
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(int(img_size * 1.05), int(img_size * 1.05)),
            A.CenterCrop(height=img_size, width=img_size, p=1.0),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

# =========================================================
# Dataset
# =========================================================
class BaseImageDataset(Dataset):
    # Image dataset used for training, validation, and test inference.
    # Training/validation returns (image, label); test returns (image, ID).
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])
        return image, int(row["label"])

# =========================================================
# Model
# =========================================================
class ArcMarginProduct(nn.Module):
    # ArcFace margin head. With labels, it applies an angular margin only to the
    # target class. Without labels, it returns margin-free scaled cosine logits
    # for inference.
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        # Compute cosine similarity between L2-normalized embeddings and class weights.
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1.0, 1.0)
        if labels is None:
            return cosine * self.s

        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # Replace only the target-class cosine value with the margin-adjusted value.
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s

class ArcFaceModel(nn.Module):
    # ConvNeXtV2 encoder with a shared embedding neck and two heads:
    #   - ce_head: standard cross-entropy classifier
    #   - arc_head: ArcFace angular-margin classifier
    def __init__(self, model_name, num_classes, embedding_dim=512, arc_s=30.0, arc_m=0.30, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(embedding_dim, num_classes, s=arc_s, m=arc_m)

    def forward(self, x, labels=None):
        feat = self.backbone(x)
        emb = self.neck(feat)
        emb = self.dropout(emb)
        ce_logits = self.ce_head(emb)
        if labels is not None:
            arc_logits = self.arc_head(emb, labels)
            return ce_logits, arc_logits, emb
        arc_logits = self.arc_head(emb, None)
        return ce_logits, arc_logits, emb

# =========================================================
# Helpers
# =========================================================
ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

def fuse_logits(ce_logits, arc_logits, infer_ce, infer_arc):
    # Normalize inference weights before combining logits. This makes the function
    # robust even if the configured weights do not sum exactly to one.
    total = infer_ce + infer_arc
    if total <= 0:
        raise ValueError("infer_ce + infer_arc must be > 0")
    infer_ce = infer_ce / total
    infer_arc = infer_arc / total
    return infer_ce * ce_logits + infer_arc * arc_logits

class LossNormalizerEMA:
    # Running loss-scale estimator for mix_mode='norm_ema'. It tracks the typical
    # magnitudes of CE loss and ArcFace loss so normalized losses can be combined.
    def __init__(self, momentum=0.95, eps=1e-8):
        self.momentum = momentum
        self.eps = eps
        self.ce_ema = None
        self.arc_ema = None

    def update(self, ce_value, arc_value):
        ce_value = float(ce_value)
        arc_value = float(arc_value)
        if self.ce_ema is None:
            self.ce_ema = ce_value
            self.arc_ema = arc_value
        else:
            self.ce_ema = self.momentum * self.ce_ema + (1 - self.momentum) * ce_value
            self.arc_ema = self.momentum * self.arc_ema + (1 - self.momentum) * arc_value

    def get(self):
        ce_scale = max(self.ce_ema if self.ce_ema is not None else 1.0, self.eps)
        arc_scale = max(self.arc_ema if self.arc_ema is not None else 1.0, self.eps)
        return ce_scale, arc_scale

def compose_training_loss(
    ce_loss,
    arc_loss,
    train_ce,
    train_arc,
    mix_mode="raw",
    arc_loss_scale=10.0,
    ema_normalizer=None
):
    # Compose a scalar training loss and return diagnostic values used for logging.
    # raw:        train_ce * ce + train_arc * arc
    # norm_fixed: train_ce * ce + train_arc * (arc / arc_loss_scale)
    # norm_ema:   train_ce * (ce / ce_ema) + train_arc * (arc / arc_ema)

    if mix_mode == "raw":
        mixed = train_ce * ce_loss + train_arc * arc_loss
        aux = {
            "ce_scale_used": 1.0,
            "arc_scale_used": 1.0,
            "ce_norm_value": float(ce_loss.detach().item()),
            "arc_norm_value": float(arc_loss.detach().item()),
        }
        return mixed, aux

    elif mix_mode == "norm_fixed":
        mixed = train_ce * ce_loss + train_arc * (arc_loss / arc_loss_scale)
        aux = {
            "ce_scale_used": 1.0,
            "arc_scale_used": float(arc_loss_scale),
            "ce_norm_value": float(ce_loss.detach().item()),
            "arc_norm_value": float((arc_loss / arc_loss_scale).detach().item()),
        }
        return mixed, aux

    elif mix_mode == "norm_ema":
        if ema_normalizer is None:
            raise ValueError("ema_normalizer is required for mix_mode='norm_ema'")

        # Use the previous EMA values to normalize the current losses.
        # The EMA is updated after the optimizer step in train_one_epoch().
        ce_scale, arc_scale = ema_normalizer.get()
        mixed = train_ce * (ce_loss / ce_scale) + train_arc * (arc_loss / arc_scale)

        aux = {
            "ce_scale_used": float(ce_scale),
            "arc_scale_used": float(arc_scale),
            "ce_norm_value": float((ce_loss / ce_scale).detach().item()),
            "arc_norm_value": float((arc_loss / arc_scale).detach().item()),
        }
        return mixed, aux

    else:
        raise ValueError(f"Unknown mix_mode: {mix_mode}")

def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    train_ce,
    train_arc,
    scaler,
    mix_mode="raw",
    arc_loss_scale=10.0,
    ema_normalizer=None
):
    # Train one epoch under a fixed training loss ratio and loss mixing mode.
    # Returned metrics include raw losses, normalized loss diagnostics, and accuracy.
    model.train()

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    total_ce_norm = total_arc_norm = 0
    total_ce_scale = total_arc_scale = 0

    arc_criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)

            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)

            loss, aux = compose_training_loss(
                ce_loss=ce_loss,
                arc_loss=arc_loss,
                train_ce=train_ce,
                train_arc=train_arc,
                mix_mode=mix_mode,
                arc_loss_scale=arc_loss_scale,
                ema_normalizer=ema_normalizer
            )

            # Training accuracy uses the same CE/ArcFace ratio as the training setting.
            fused_logits = fuse_logits(ce_logits, arc_logits, train_ce, train_arc)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Update EMA normalizer after the step using raw loss values.
        if mix_mode == "norm_ema" and ema_normalizer is not None:
            ema_normalizer.update(
                ce_value=ce_loss.detach().item(),
                arc_value=arc_loss.detach().item()
            )

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_ce += ce_loss.item() * bs
        total_arc += arc_loss.item() * bs
        total_ce_norm += aux["ce_norm_value"] * bs
        total_arc_norm += aux["arc_norm_value"] * bs
        total_ce_scale += aux["ce_scale_used"] * bs
        total_arc_scale += aux["arc_scale_used"] * bs
        total_correct += (fused_logits.argmax(1) == labels).sum().item()
        total_count += bs

    return {
        "loss": total_loss / total_count,
        "ce_loss": total_ce / total_count,
        "arc_loss": total_arc / total_count,
        "ce_norm": total_ce_norm / total_count,
        "arc_norm": total_arc_norm / total_count,
        "ce_scale": total_ce_scale / total_count,
        "arc_scale": total_arc_scale / total_count,
        "acc": total_correct / total_count,
    }

@torch.no_grad()
def valid_one_epoch(model, loader, device, train_ce, train_arc, infer_ce, infer_arc):
    # Deterministic validation pass during training. The validation loss follows
    # the training ratio, while validation accuracy uses the specified inference ratio.
    model.eval()
    arc_criterion = nn.CrossEntropyLoss()

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)

            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            # Keep validation loss as raw train-objective-style scalar for reference.
            loss = train_ce * ce_loss + train_arc * arc_loss

            fused_logits = fuse_logits(ce_logits, arc_logits, infer_ce, infer_arc)
            probs = torch.softmax(fused_logits, dim=1)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_ce += ce_loss.item() * bs
        total_arc += arc_loss.item() * bs
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += bs
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_labels, axis=0),
    )

@torch.no_grad()
def predict_probs(model, loader, device, infer_ce, infer_arc):
    # Predict probabilities for one DataLoader using inference-mode ArcFace logits
    # and the requested CE/ArcFace logit fusion ratio.
    model.eval()
    all_probs, all_ids = [], []

    for images, ids in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels=None)
            logits = fuse_logits(ce_logits, arc_logits, infer_ce, infer_arc)
            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_ids.extend(ids.tolist() if torch.is_tensor(ids) else list(ids))

    return np.array(all_ids), np.concatenate(all_probs, axis=0)

@torch.no_grad()
def predict_with_tta(model, df, device, infer_ce, infer_arc, is_test=True, tta_count=4):
    # Run all configured TTA views and average their probabilities. For validation,
    # labels are returned; for test inference, image IDs are returned.
    probs_list, labels_ref, ids_ref = [], None, None

    for tta_id in range(tta_count):
        ds = BaseImageDataset(
            df,
            transform=get_tta_transforms(cfg.image_size, tta_id),
            is_test=is_test
        )
        dl = DataLoader(
            ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False
        )

        if is_test:
            ids, probs = predict_probs(model, dl, device, infer_ce, infer_arc)
            ids_ref = ids
            probs_list.append(probs)
        else:
            fold_probs, fold_labels = [], []
            for images, labels in dl:
                images = images.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    ce_logits, arc_logits, _ = model(images, labels=None)
                    logits = fuse_logits(ce_logits, arc_logits, infer_ce, infer_arc)
                    probs = torch.softmax(logits, dim=1)

                fold_probs.append(probs.cpu().numpy())
                fold_labels.append(labels.numpy())

            labels_ref = np.concatenate(fold_labels, axis=0)
            probs_list.append(np.concatenate(fold_probs, axis=0))

        del ds, dl
        gc.collect()

    mean_probs = np.mean(probs_list, axis=0)
    return (ids_ref, mean_probs) if is_test else (mean_probs, labels_ref)

# =========================================================
# CV splits
# =========================================================
# Precompute stratified folds so every train/infer/mixing configuration uses
# identical splits.
skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
splits = list(skf.split(train_df, train_df["label"]))

# =========================================================
# Main sweep
# =========================================================
summary_rows = []

# Main experiment loop:
# 1. Select a loss mixing mode.
# 2. Train fold-wise models for each training CE/ArcFace ratio.
# 3. Reuse those checkpoints to evaluate each inference CE/ArcFace ratio.
for mix_setting in MIXING_MODES:
    mix_tag = mix_setting["mix_tag"]
    mix_mode = mix_setting["mix_mode"]
    arc_loss_scale = mix_setting["arc_loss_scale"]
    ema_momentum = mix_setting["ema_momentum"]

    print("\n" + "=" * 120)
    print(f"MIXING MODE: {mix_tag}")
    print(f"mix_mode={mix_mode}, arc_loss_scale={arc_loss_scale}, ema_momentum={ema_momentum}")
    print("=" * 120)

    for train_setting in TRAIN_SETTINGS:
        train_name = train_setting["name"]
        train_ce = train_setting["train_ce"]
        train_arc = train_setting["train_arc"]

        print("\n" + "#" * 120)
        print(f"TRAIN SETTING: {train_name} | MIX: {mix_tag}")
        print(f"train_ce={train_ce}, train_arc={train_arc}")
        print("#" * 120)

        fold_model_paths = []
        fold_val_dfs = []
        fold_va_indices = []
        fold_best_valid_accs = []
        fold_train_logs = []

        # ---------------------------------------------
        # Phase 1: train once per fold
        # ---------------------------------------------
        # The checkpoints trained here are later reused for all inference ratios.
        for fold, (tr_idx, va_idx) in enumerate(splits):
            print("\n" + "=" * 90)
            print(f"TRAINING FOLD {fold+1}/{cfg.n_splits}")
            print("=" * 90)

            trn_df = train_df.iloc[tr_idx].reset_index(drop=True)
            val_df = train_df.iloc[va_idx].reset_index(drop=True)

            train_ds = BaseImageDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False)
            valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

            train_loader = DataLoader(
                train_ds,
                batch_size=cfg.train_bs,
                shuffle=True,
                num_workers=cfg.num_workers,
                pin_memory=True
            )
            valid_loader = DataLoader(
                valid_ds,
                batch_size=cfg.valid_bs,
                shuffle=False,
                num_workers=cfg.num_workers,
                pin_memory=True
            )

            model = ArcFaceModel(
                cfg.model_name,
                cfg.num_classes,
                embedding_dim=cfg.embedding_dim,
                arc_s=cfg.arc_s,
                arc_m=cfg.arc_m,
                dropout=cfg.dropout
            ).to(cfg.device)

            backbone_params = list(model.backbone.parameters())
            head_params = list(model.neck.parameters()) + list(model.ce_head.parameters()) + list(model.arc_head.parameters())

            optimizer = torch.optim.AdamW(
                [{"params": backbone_params, "lr": cfg.lr_backbone},
                 {"params": head_params, "lr": cfg.lr_head}],
                weight_decay=cfg.weight_decay
            )

            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg.epochs * len(train_loader),
                eta_min=cfg.min_lr
            )

            scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
            stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")

            ema_normalizer = None
            if mix_mode == "norm_ema":
                ema_normalizer = LossNormalizerEMA(momentum=ema_momentum)

            best_acc = -1.0
            best_path = best_model_path(train_name, mix_tag, fold)
            fold_epoch_logs = []

            for epoch in range(cfg.epochs):
                t0 = time.time()

                # During training, the early-stopping target uses infer=train.
                tr_stats = train_one_epoch(
                    model=model,
                    loader=train_loader,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    device=cfg.device,
                    train_ce=train_ce,
                    train_arc=train_arc,
                    scaler=scaler,
                    mix_mode=mix_mode,
                    arc_loss_scale=arc_loss_scale,
                    ema_normalizer=ema_normalizer
                )

                va_loss, va_ce_loss, va_arc_loss, va_acc, _, _ = valid_one_epoch(
                    model=model,
                    loader=valid_loader,
                    device=cfg.device,
                    train_ce=train_ce,
                    train_arc=train_arc,
                    infer_ce=train_ce,
                    infer_arc=train_arc
                )

                print(
                    f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                    f"train_loss={tr_stats['loss']:.4f} "
                    f"(ce={tr_stats['ce_loss']:.4f}, arc={tr_stats['arc_loss']:.4f}, "
                    f"ce_norm={tr_stats['ce_norm']:.4f}, arc_norm={tr_stats['arc_norm']:.4f}) | "
                    f"train_acc={tr_stats['acc']:.4f} | "
                    f"valid_loss={va_loss:.4f} (ce={va_ce_loss:.4f}, arc={va_arc_loss:.4f}) | "
                    f"valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
                )

                fold_epoch_logs.append({
                    "epoch": epoch + 1,
                    "train_loss": float(tr_stats["loss"]),
                    "train_ce_loss": float(tr_stats["ce_loss"]),
                    "train_arc_loss": float(tr_stats["arc_loss"]),
                    "train_ce_norm": float(tr_stats["ce_norm"]),
                    "train_arc_norm": float(tr_stats["arc_norm"]),
                    "train_ce_scale": float(tr_stats["ce_scale"]),
                    "train_arc_scale": float(tr_stats["arc_scale"]),
                    "train_acc": float(tr_stats["acc"]),
                    "valid_loss": float(va_loss),
                    "valid_ce_loss": float(va_ce_loss),
                    "valid_arc_loss": float(va_arc_loss),
                    "valid_acc": float(va_acc),
                })

                if va_acc > best_acc:
                    best_acc = va_acc
                    torch.save(model.state_dict(), best_path)
                    print(f"  saved best -> {best_path}")

                if stopper.step(va_acc):
                    print(f"  early stopping at epoch {epoch+1}")
                    break

            fold_model_paths.append(best_path)
            fold_val_dfs.append(val_df.copy())
            fold_va_indices.append(va_idx.copy())
            fold_best_valid_accs.append(float(best_acc))
            fold_train_logs.append(fold_epoch_logs)

            del model, optimizer, scheduler, scaler, train_loader, valid_loader, train_ds, valid_ds
            gc.collect()
            torch.cuda.empty_cache()

        # Save metadata for the training phase before evaluating inference settings.
        train_meta = {
            "train_setting": train_setting,
            "mix_setting": mix_setting,
            "cfg": asdict(cfg),
            "fold_best_valid_accs": fold_best_valid_accs,
            "mean_best_valid_acc": float(np.mean(fold_best_valid_accs)),
            "fold_train_logs": fold_train_logs,
        }
        with open(train_metadata_path(train_name, mix_tag), "w", encoding="utf-8") as f:
            json.dump(train_meta, f, ensure_ascii=False, indent=2)

        # ---------------------------------------------
        # Phase 2: evaluate all infer settings
        # ---------------------------------------------
        # Evaluation reloads each saved fold checkpoint and recomputes OOF/test
        # probabilities for each inference-time fusion ratio.
        for infer_setting in INFER_SETTINGS:
            infer_name = infer_setting["name"]
            infer_ce = infer_setting["infer_ce"]
            infer_arc = infer_setting["infer_arc"]

            print("\n" + "-" * 120)
            print(f"EVALUATING train={train_name} | mix={mix_tag} | infer={infer_name}")
            print(f"infer_ce={infer_ce}, infer_arc={infer_arc}")
            print("-" * 120)

            oof_probs = np.zeros((len(train_df), cfg.num_classes), dtype=np.float32)
            test_probs = np.zeros((len(test_df), cfg.num_classes), dtype=np.float32)
            fold_scores = []

            for fold in range(cfg.n_splits):
                print(f"[Eval] fold {fold+1}/{cfg.n_splits}")

                va_idx = fold_va_indices[fold]
                val_df = fold_val_dfs[fold]

                model = ArcFaceModel(
                    cfg.model_name,
                    cfg.num_classes,
                    embedding_dim=cfg.embedding_dim,
                    arc_s=cfg.arc_s,
                    arc_m=cfg.arc_m,
                    dropout=cfg.dropout
                ).to(cfg.device)

                model.load_state_dict(torch.load(fold_model_paths[fold], map_location=cfg.device))

                val_tta_probs, val_tta_labels = predict_with_tta(
                    model, val_df, cfg.device,
                    infer_ce=infer_ce, infer_arc=infer_arc,
                    is_test=False, tta_count=cfg.tta_count
                )
                val_tta_acc = accuracy_score(val_tta_labels, val_tta_probs.argmax(1))
                print(f"  fold {fold+1} TTA valid acc: {val_tta_acc:.5f}")

                oof_probs[va_idx] = val_tta_probs

                _, fold_test_probs = predict_with_tta(
                    model, test_df, cfg.device,
                    infer_ce=infer_ce, infer_arc=infer_arc,
                    is_test=True, tta_count=cfg.tta_count
                )
                test_probs += fold_test_probs / cfg.n_splits

                fold_scores.append({
                    "fold": fold,
                    "best_valid_acc_trainphase": float(fold_best_valid_accs[fold]),
                    "tta_valid_acc_evalphase": float(val_tta_acc),
                })

                del model
                gc.collect()
                torch.cuda.empty_cache()

            oof_preds = oof_probs.argmax(axis=1)
            oof_acc = accuracy_score(train_df["label"].values, oof_preds)

            np.save(infer_oof_probs_path(train_name, mix_tag, infer_name), oof_probs)
            np.save(infer_test_probs_path(train_name, mix_tag, infer_name), test_probs)

            submission = test_df[["ID"]].copy()
            submission["TARGET"] = test_probs.argmax(axis=1).astype(int)
            submission.to_csv(infer_submission_path(train_name, mix_tag, infer_name), index=False)

            fold_scores_df = pd.DataFrame(fold_scores)
            fold_scores_df.to_csv(infer_fold_scores_path(train_name, mix_tag, infer_name), index=False)

            result_row = {
                "mix_tag": mix_tag,
                "mix_mode": mix_mode,
                "arc_loss_scale": arc_loss_scale,
                "ema_momentum": ema_momentum,
                "train_name": train_name,
                "train_ce": train_ce,
                "train_arc": train_arc,
                "infer_name": infer_name,
                "infer_ce": infer_ce,
                "infer_arc": infer_arc,
                "oof_accuracy": float(oof_acc),
                "mean_tta_valid_acc": float(fold_scores_df["tta_valid_acc_evalphase"].mean()),
                "std_tta_valid_acc": float(fold_scores_df["tta_valid_acc_evalphase"].std(ddof=0)),
                "mean_best_valid_acc_trainphase": float(np.mean(fold_best_valid_accs)),
            }
            summary_rows.append(result_row)

            print(
                f"[Result] mix={mix_tag} | train={train_name} | infer={infer_name} | "
                f"OOF={oof_acc:.6f} | mean_TTA={result_row['mean_tta_valid_acc']:.6f}"
            )

# Aggregate all experiment results and sort by OOF accuracy first.
summary_df = pd.DataFrame(summary_rows).sort_values(
    ["oof_accuracy", "mean_tta_valid_acc"],
    ascending=[False, False]
).reset_index(drop=True)

summary_path = os.path.join(SAVE_DIR, "traingrid_norm_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "#" * 120)
print("TRAIN-GRID × INFER-GRID × MIXING-MODE SUMMARY")
print(summary_df.head(50))
print(f"saved -> {summary_path}")
print("#" * 120)


### Post-deadline diagnostic ensemble

This optional cell reproduces the post-deadline diagnostic ensembles discussed in the paper.
These submissions were **not** part of the official final ranking.
They are provided only to analyze the public/private discrepancy after the private leaderboard was released.


In [ ]:
# =========================================================
# Post-deadline diagnostic ensemble generation
# ---------------------------------------------------------
# These ensembles were NOT part of the official final submissions.
# They are provided only to reproduce the retrospective diagnostic
# analysis discussed in the paper.
#
# Diagnostic D1:
#   0.700 train_ce20_arc80_raw_infer_ce50_arc50
#   0.200 ssl_pre2_base
#   0.100 convnextv2_base_512_arcface_ce60arc40
#
# Diagnostic D2:
#   0.700 train_ce20_arc80_raw_infer_ce50_arc50
#   0.200 ssl_pre2_base
#   0.100 arcface_base_old_ft768_v1
#
# Diagnostic D3:
#   0.800 train_ce20_arc80_raw_infer_ce50_arc50
#   0.200 ssl_pre2_base
#
# Required files:
#   ./ensemble_artifacts/{prefix}_test_probs.npy
#   ./Data/test.csv
#
# Outputs:
#   ./ensemble_artifacts/postdiag_ce20arc80_ssl_512_test_probs.npy
#   ./ensemble_artifacts/postdiag_ce20arc80_ssl_512_submission.csv
#   ./ensemble_artifacts/postdiag_ce20arc80_ssl_768_test_probs.npy
#   ./ensemble_artifacts/postdiag_ce20arc80_ssl_768_submission.csv
#   ./ensemble_artifacts/postdiag_ce20arc80_ssl_test_probs.npy
#   ./ensemble_artifacts/postdiag_ce20arc80_ssl_submission.csv
#   ./ensemble_artifacts/post_deadline_diagnostic_summary.csv
# =========================================================

import os
import json
import numpy as np
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
SAVE_DIR = "./ensemble_artifacts"
TEST_CSV = "./Data/test.csv"

os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Post-deadline diagnostic ensemble definitions
# -----------------------------
POST_DEADLINE_DIAGNOSTIC_ENSEMBLES = [
    {
        "name": "postdiag_ce20arc80_ssl_512",
        "description": "0.700 CE20/Arc80 diagnostic branch + 0.200 SSL + 0.100 512 CE+ArcFace",
        "prefixes": [
            "train_ce20_arc80_raw_infer_ce50_arc50",
            "ssl_pre2_base",
            "convnextv2_base_512_arcface_ce60arc40",
        ],
        "weights": [0.700, 0.200, 0.100],
        "public_lb": 0.994666,
        "private_lb": 0.996000,
        "official_final": False,
    },
    {
        "name": "postdiag_ce20arc80_ssl_768",
        "description": "0.700 CE20/Arc80 diagnostic branch + 0.200 SSL + 0.100 768 CE+ArcFace",
        "prefixes": [
            "train_ce20_arc80_raw_infer_ce50_arc50",
            "ssl_pre2_base",
            "arcface_base_old_ft768_v1",
        ],
        "weights": [0.700, 0.200, 0.100],
        "public_lb": 0.994666,
        "private_lb": 0.996000,
        "official_final": False,
    },
    {
        "name": "postdiag_ce20arc80_ssl",
        "description": "0.800 CE20/Arc80 diagnostic branch + 0.200 SSL",
        "prefixes": [
            "train_ce20_arc80_raw_infer_ce50_arc50",
            "ssl_pre2_base",
        ],
        "weights": [0.800, 0.200],
        "public_lb": 0.994666,
        "private_lb": 0.996000,
        "official_final": False,
    },
]

# -----------------------------
# Load test IDs
# -----------------------------
test_df = pd.read_csv(TEST_CSV)

if "ID" not in test_df.columns:
    raise ValueError("test.csv must contain an 'ID' column.")

print("test_df:", test_df.shape)
print("SAVE_DIR:", SAVE_DIR)

# -----------------------------
# Helper functions
# -----------------------------
def load_probs(prefix: str, save_dir: str = SAVE_DIR) -> np.ndarray:
    """Load test probability file for one model prefix."""
    prefix = prefix.strip()
    path = os.path.join(save_dir, f"{prefix}_test_probs.npy")

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing probability file: {path}\n"
            f"Expected file name pattern: {{prefix}}_test_probs.npy"
        )

    probs = np.load(path)

    if probs.ndim != 2:
        raise ValueError(f"{path} must be 2D, but got shape {probs.shape}")

    return probs.astype(np.float32)


def make_weighted_ensemble(prefixes, weights):
    """Create a weighted average of probability arrays."""
    prefixes = [p.strip() for p in prefixes]
    weights = np.asarray(weights, dtype=np.float32)

    if len(prefixes) != len(weights):
        raise ValueError(
            f"len(prefixes)={len(prefixes)} and len(weights)={len(weights)} must match."
        )

    weights = weights / weights.sum()

    prob_list = []
    for prefix in prefixes:
        probs = load_probs(prefix)
        print(f"  loaded {prefix}: shape={probs.shape}, dtype={probs.dtype}")
        prob_list.append(probs)

    base_shape = prob_list[0].shape
    for prefix, probs in zip(prefixes, prob_list):
        if probs.shape != base_shape:
            raise ValueError(
                f"Shape mismatch: {prefix} has {probs.shape}, expected {base_shape}"
            )

    if base_shape[0] != len(test_df):
        raise ValueError(
            f"Number of test rows mismatch: probs has {base_shape[0]}, "
            f"but test_df has {len(test_df)}"
        )

    ensemble_probs = np.zeros(base_shape, dtype=np.float32)
    for w, probs in zip(weights, prob_list):
        ensemble_probs += w * probs

    # Optional safety check: probabilities should be approximately normalized.
    row_sums = ensemble_probs.sum(axis=1)
    print(
        f"  ensemble row-sum: min={row_sums.min():.6f}, "
        f"max={row_sums.max():.6f}, mean={row_sums.mean():.6f}"
    )

    return ensemble_probs, weights


# -----------------------------
# Generate post-deadline diagnostic submissions
# -----------------------------
summary_rows = []

for ens in POST_DEADLINE_DIAGNOSTIC_ENSEMBLES:
    print("\n" + "=" * 100)
    print(ens["name"])
    print(ens["description"])
    print("NOTE: This is a post-deadline diagnostic ensemble, not an official final submission.")
    print("=" * 100)

    prefixes = ens["prefixes"]
    weights = ens["weights"]

    print("prefixes:", prefixes)
    print("weights :", weights)

    ensemble_probs, normalized_weights = make_weighted_ensemble(prefixes, weights)
    preds = ensemble_probs.argmax(axis=1).astype(int)

    submission = test_df[["ID"]].copy()
    submission["TARGET"] = preds

    probs_out = os.path.join(SAVE_DIR, f"{ens['name']}_test_probs.npy")
    sub_out = os.path.join(SAVE_DIR, f"{ens['name']}_submission.csv")
    meta_out = os.path.join(SAVE_DIR, f"{ens['name']}_metadata.json")

    np.save(probs_out, ensemble_probs)
    submission.to_csv(sub_out, index=False)

    metadata = {
        "name": ens["name"],
        "description": ens["description"],
        "prefixes": prefixes,
        "weights": [float(w) for w in normalized_weights],
        "public_lb": ens["public_lb"],
        "private_lb": ens["private_lb"],
        "official_final": ens["official_final"],
        "note": "Post-deadline diagnostic submission. Not included in the official final ranking.",
        "probs_out": probs_out,
        "submission_out": sub_out,
    }

    with open(meta_out, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"saved probs      -> {probs_out}")
    print(f"saved submission -> {sub_out}")
    print(f"saved metadata   -> {meta_out}")
    print(submission.head())

    summary_rows.append({
        "name": ens["name"],
        "description": ens["description"],
        "prefixes": " + ".join(prefixes),
        "weights": " + ".join([f"{w:.3f}" for w in normalized_weights]),
        "public_lb": ens["public_lb"],
        "private_lb": ens["private_lb"],
        "official_final": ens["official_final"],
        "submission_file": os.path.basename(sub_out),
        "probs_file": os.path.basename(probs_out),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(SAVE_DIR, "post_deadline_diagnostic_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 100)
print("Post-deadline diagnostic summary")
print("=" * 100)
print(summary_df)
print(f"saved summary -> {summary_path}")
